# TSP Optimization Test Notebook
Este notebook prueba el flujo de obtención de la matriz de distancias y la resolución del TSP usando OR-Tools.

In [4]:
import os
import requests
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

In [ ]:
# Configuración de parámetros
API_KEY = '' #os.getenv('GOOGLE_MAPS_API_KEY')
points = [
    (43.7781037, 11.2588956),
    (43.7767975, 11.2540480),
    (43.7752073, 11.2489769),
]

In [6]:
# Obtener la matriz de distancias
origins = '|'.join(f"{lat},{lng}" for lat, lng in points)
destinations = origins
url = (
    f"https://maps.googleapis.com/maps/api/distancematrix/json?"
    f"origins={origins}&destinations={destinations}&mode=walking&key={API_KEY}"
)
response = requests.get(url)
data = response.json()
matrix = [
    [elem['distance']['value'] for elem in row['elements']]
    for row in data['rows']
]
matrix

[[0, 576, 1092], [576, 0, 557], [1092, 557, 0]]

In [10]:
# Resolver el TSP con OR-Tools
n = len(matrix)
manager = pywrapcp.RoutingIndexManager(n, 1, 0)
routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return matrix[from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

search_params = pywrapcp.DefaultRoutingSearchParameters()
search_params.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)

solution = routing.SolveWithParameters(search_params)

# Extraer la ruta óptima
route = []
index = routing.Start(0)
while not routing.IsEnd(index):
    route.append(manager.IndexToNode(index))
    index = solution.Value(routing.NextVar(index))
route.append(manager.IndexToNode(index))
route = route[:-1]

In [12]:
# Construir URL de Google Maps para visualizar la ruta
ordered_points = [points[i] for i in route]
origin = ordered_points[0]
destination = ordered_points[-1]
waypoints = ordered_points[1:-1]
waypoints_str = '|'.join(f"{lat},{lng}" for lat, lng in waypoints)

maps_url = (
    f"https://www.google.com/maps/dir/?api=1&origin={origin[0]},{origin[1]}"
    f"&destination={destination[0]},{destination[1]}&travelmode=walking&waypoints={waypoints_str}"
)
maps_url

'https://www.google.com/maps/dir/?api=1&origin=43.7781037,11.2588956&destination=43.7752073,11.2489769&travelmode=walking&waypoints=43.7767975,11.254048'